In [1]:
import pandas as pd
import numpy as np
from IPython.display import Math

In [2]:
training_set = pd.read_csv('/home/lukas/Desktop/digit-recognizer/train.csv')
test = pd.read_csv('/home/lukas/Desktop/digit-recognizer/test.csv')

In [67]:
train_set = training_set.to_numpy()
y = training_set['label'].to_numpy()

In [115]:
rng = np.random.default_rng()

In [114]:
def create_batches(arr, batch_size):
    rng.shuffle(arr)
    container = []
    labels = []
    
    for i in range(0, len(arr), batch_size):
        sample = arr[i : i + batch_size, 1:]
        label = arr[i : i + batch_size, 0]
        labels.append(label)
        container.append(sample)

    return container, labels

In [36]:
def ReLU(x):
    x = np.maximum(0, x)
    return x

In [37]:
def Softmax(x):
    x_exp = np.exp(x)
    output = x_exp / np.sum(x_exp)

    return output

In [38]:
def layer(weights, bias, x_in, func):
    
    z = x_in @ weights + bias
    activation = func(z)

    return z, activation

In [39]:
def init_parameters(layer_neurons):
    num_layers = len(layer_neurons)
    list_weights = []
    list_bias = []
    
    for i in range(1, num_layers):
        n_in = layer_neurons[i - 1]
        n_out = layer_neurons[i]

        weight = np.random.randn(n_in, n_out) * np.sqrt(2 / n_in)
        list_weights.append(weight)
        list_bias.append(np.zeros((1, n_out)))

    return list_weights, list_bias

In [40]:
layer_neurons = [784, 10, 10, 10]
w, b = init_parameters(layer_neurons)

In [41]:
def forward_prop(x_in, w, b):
    num_layers = len(w)
    activation_dict = {}
    z_dict = {}
    
    for i in range(num_layers - 1):
        z, a = layer(w[i], b[i], x_in, ReLU)
        x_in = a
        z_dict[i + 1] = z
        activation_dict[i + 1] = a
        
    z, a = layer(w[-1], b[-1], z, Softmax)
    z_dict[num_layers] = z
    activation_dict[num_layers] = a
    
    return activation_dict, z_dict

In [44]:
a_dict, z_dict = forward_prop(x_in, w, b)

In [45]:
def compute_cost(x_in, label):
    
    idx = np.argmax(label)            
    x = x_in[0]
    
    return -np.log(x[idx] + 1e-12)   

In [46]:
def ReLU_derivative(z):
    return (z > 0).astype(float)

In [50]:

def backpropagation(w, b, activation_dict, z_dict, y, x_in):
    num_operations = len(w)
    gradients_w = [None] * num_operations
    gradients_b = [None] * num_operations

    ###  Pochodna dC/dw
    A_out = activation_dict[num_operations] ### Funkcja aktywacji którą wyrzuca output layer
    A_prev = activation_dict[num_operations - 1] ### Funkcja aktywacji która trafia do output layer

    # dC/da(L)
    gamma = (A_out - y) * ReLU_derivative(z_dict[num_operations])

    gradients_w[-1] = A_prev.T @ gamma
    gradients_b[-1] = np.sum(gamma, axis = 0, keepdims=True)


    for L in reversed(range(1, num_operations)):

        if L == 1:
            A_prev = x_in
        else:
            A_prev = activation_dict[L]

        # gamma z następnej warstwy * waga następnej warstwy * pochodna funkcji aktywacji aktualnej warstwy
        gamma = (gamma @ w[L].T) * ReLU_derivative(z_dict[L])
        gradients_w[L-1] = A_prev.T @ gamma
        gradients_b[L-1] = np.sum(gamma, axis = 0, keepdims=True)


    return gradients_w, gradients_b
    

In [51]:
x_in = np.random.randn(1, 784)
label = np.zeros((1, 10))

In [52]:
gradients_w, gradients_b = backpropagation(w, b, a_dict, z_dict, label, x_in)

In [53]:
w[0]

array([[-0.03543817,  0.01129729, -0.02020364, ...,  0.03261887,
        -0.01347121, -0.01276019],
       [ 0.03954414, -0.09384572, -0.00664205, ..., -0.07480619,
        -0.02890351,  0.02490254],
       [ 0.01084601,  0.00723114,  0.01670839, ..., -0.03573636,
        -0.0370455 , -0.01438064],
       ...,
       [ 0.0199933 ,  0.02635925, -0.02457678, ...,  0.10554163,
         0.01078782, -0.00910845],
       [ 0.04324568,  0.07894808, -0.09753234, ..., -0.05621134,
         0.00694323,  0.00381541],
       [-0.03195484, -0.03274643, -0.03419908, ..., -0.02083536,
         0.00989296,  0.04617542]], shape=(784, 10))